## SRP404822

**paper:** [PMID: 36548828](https://pmc.ncbi.nlm.nih.gov/articles/PMC9784451/) - Using Blood Transcriptome Analysis to Determine the Changes in Immunity and Metabolism of Giant Pandas with Age, 2022

**date, curator:** 2026-04-08, Sara Carsanaro

**notes**

### annotation summary

In [21]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,Blood,UBERON:0000178,blood,perfect match


In [22]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,3,UBERON:0034919,juvenile stage,missing child term
3,4,UBERON:0034919,juvenile stage,missing child term
6,5,UBERON:0034919,juvenile stage,missing child term
9,6,UBERON:0018241,prime adult stage,missing child term
13,7,UBERON:0018241,prime adult stage,missing child term
15,8,UBERON:0018241,prime adult stage,missing child term
18,9,UBERON:0018241,prime adult stage,missing child term
19,10,UBERON:0018241,prime adult stage,missing child term
21,11,UBERON:0018241,prime adult stage,missing child term
24,12,UBERON:0018241,prime adult stage,missing child term


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP404822"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
100%|███████████████████████████████████████████| 48/48 [00:48<00:00,  1.00s/it]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX18043695,SRP404822,MGISEQ-2000RS,SRS15550289,,,UBERON:0034919,juvenile stage,,Blood,3,,,missing child term,F,,,9646,,,,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476497,3,,,,,,,,,,08/04/2026,health,,3,,,,,,TRANSCRIPTOMIC,cDNA
1,SRX18043684,SRP404822,MGISEQ-2000RS,SRS15550277,,,UBERON:0034919,juvenile stage,,Blood,3,,,missing child term,F,,,9646,,,,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476496,3,,,,,,,,,,08/04/2026,health,,2,,,,,,TRANSCRIPTOMIC,cDNA
2,SRX18043683,SRP404822,MGISEQ-2000RS,SRS15550278,,,UBERON:0034919,juvenile stage,,Blood,3,,,missing child term,M,,,9646,,,,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476495,3,,,,,,,,,,08/04/2026,health,,1,,,,,,TRANSCRIPTOMIC,cDNA
3,SRX18043726,SRP404822,MGISEQ-2000RS,SRS15550320,,,UBERON:0034919,juvenile stage,,Blood,4,,,missing child term,M,,,9646,,,,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476500,4,,,,,,,,,,08/04/2026,health,,6,,,,,,TRANSCRIPTOMIC,cDNA
4,SRX18043717,SRP404822,MGISEQ-2000RS,SRS15550311,,,UBERON:0034919,juvenile stage,,Blood,4,,,missing child term,F,,,9646,,,,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476499,4,,,,,,,,,,08/04/2026,health,,5,,,,,,TRANSCRIPTOMIC,cDNA
5,SRX18043706,SRP404822,MGISEQ-2000RS,SRS15550300,,,UBERON:0034919,juvenile stage,,Blood,4,,,missing child term,M,,,9646,,,,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476498,4,,,,,,,,,,08/04/2026,health,,4,,,,,,TRANSCRIPTOMIC,cDNA
6,SRX18043729,SRP404822,MGISEQ-2000RS,SRS15550323,,,UBERON:0034919,juvenile stage,,Blood,5,,,missing child term,F,,,9646,,,,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476503,5,,,,,,,,,,08/04/2026,health,,9,,,,,,TRANSCRIPTOMIC,cDNA
7,SRX18043728,SRP404822,MGISEQ-2000RS,SRS15550322,,,UBERON:0034919,juvenile stage,,Blood,5,,,missing child term,F,,,9646,,,,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476502,5,,,,,,,,,,08/04/2026,health,,8,,,,,,TRANSCRIPTOMIC,cDNA
8,SRX18043727,SRP404822,MGISEQ-2000RS,SRS15550321,,,UBERON:0034919,juvenile stage,,Blood,5,,,missing child term,F,,,9646,,,,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476501,5,,,,,,,,,,08/04/2026,health,,7,,,,,,TRANSCRIPTOMIC,cDNA
9,SRX18043730,SRP404822,MGISEQ-2000RS,SRS15550324,,,UBERON:0018241,prime adult stage,,Blood,6,,,missing child term,F,,,9646,,,,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476504,6,,,,,,,,,,08/04/2026,health,,10,,,,,,TRANSCRIPTOMIC,cDNA


#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['Blood']


In [6]:

# all
library.loc[:,'anatId'] = 'UBERON:0000178'
library.loc[:,'anatName'] = 'blood'
# perfect match, missing child term, other
library.loc[:,'anatAnnotationStatus'] = 'perfect match'
# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'not documented'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX18043695,SRP404822,MGISEQ-2000RS,SRS15550289,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,3,perfect match,not documented,missing child term,F,,,9646,,,,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476497,3,,,,,,,,,,08/04/2026,health,,3,,,,,,TRANSCRIPTOMIC,cDNA
1,SRX18043684,SRP404822,MGISEQ-2000RS,SRS15550277,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,3,perfect match,not documented,missing child term,F,,,9646,,,,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476496,3,,,,,,,,,,08/04/2026,health,,2,,,,,,TRANSCRIPTOMIC,cDNA
2,SRX18043683,SRP404822,MGISEQ-2000RS,SRS15550278,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,3,perfect match,not documented,missing child term,M,,,9646,,,,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476495,3,,,,,,,,,,08/04/2026,health,,1,,,,,,TRANSCRIPTOMIC,cDNA
3,SRX18043726,SRP404822,MGISEQ-2000RS,SRS15550320,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,4,perfect match,not documented,missing child term,M,,,9646,,,,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476500,4,,,,,,,,,,08/04/2026,health,,6,,,,,,TRANSCRIPTOMIC,cDNA
4,SRX18043717,SRP404822,MGISEQ-2000RS,SRS15550311,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,4,perfect match,not documented,missing child term,F,,,9646,,,,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476499,4,,,,,,,,,,08/04/2026,health,,5,,,,,,TRANSCRIPTOMIC,cDNA
5,SRX18043706,SRP404822,MGISEQ-2000RS,SRS15550300,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,4,perfect match,not documented,missing child term,M,,,9646,,,,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476498,4,,,,,,,,,,08/04/2026,health,,4,,,,,,TRANSCRIPTOMIC,cDNA
6,SRX18043729,SRP404822,MGISEQ-2000RS,SRS15550323,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,5,perfect match,not documented,missing child term,F,,,9646,,,,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476503,5,,,,,,,,,,08/04/2026,health,,9,,,,,,TRANSCRIPTOMIC,cDNA
7,SRX18043728,SRP404822,MGISEQ-2000RS,SRS15550322,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,5,perfect match,not documented,missing child term,F,,,9646,,,,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476502,5,,,,,,,,,,08/04/2026,health,,8,,,,,,TRANSCRIPTOMIC,cDNA
8,SRX18043727,SRP404822,MGISEQ-2000RS,SRS15550321,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,5,perfect match,not documented,missing child term,F,,,9646,,,,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476501,5,,,,,,,,,,08/04/2026,health,,7,,,,,,TRANSCRIPTOMIC,cDNA
9,SRX18043730,SRP404822,MGISEQ-2000RS,SRS15550324,UBERON:0000178,blood,UBERON:0018241,prime adult stage,,Blood,6,perfect match,not documented,missing child term,F,,,9646,,,,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476504,6,,,,,,,,,,08/04/2026,health,,10,,,,,,TRANSCRIPTOMIC,cDNA


#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['10' '11' '12' '13' '14' '15' '17' '18' '19' '21' '22' '23' '26' '27'
 '28' '3' '30' '4' '5' '6' '7' '8' '9']


#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [8]:
# making these variables because we use them again in the experiment file
#my_protocol = ''
# full_length or 3'
#my_protocolType = ''

#library.loc[:,'protocol'] = my_protocol
#library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX18043695,SRP404822,MGISEQ-2000RS,SRS15550289,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,3,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476497,3,,,,,,,,,,08/04/2026,health,,3,,,,,,TRANSCRIPTOMIC,cDNA
1,SRX18043684,SRP404822,MGISEQ-2000RS,SRS15550277,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,3,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476496,3,,,,,,,,,,08/04/2026,health,,2,,,,,,TRANSCRIPTOMIC,cDNA
2,SRX18043683,SRP404822,MGISEQ-2000RS,SRS15550278,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,3,perfect match,not documented,missing child term,M,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476495,3,,,,,,,,,,08/04/2026,health,,1,,,,,,TRANSCRIPTOMIC,cDNA
3,SRX18043726,SRP404822,MGISEQ-2000RS,SRS15550320,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,4,perfect match,not documented,missing child term,M,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476500,4,,,,,,,,,,08/04/2026,health,,6,,,,,,TRANSCRIPTOMIC,cDNA
4,SRX18043717,SRP404822,MGISEQ-2000RS,SRS15550311,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,4,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476499,4,,,,,,,,,,08/04/2026,health,,5,,,,,,TRANSCRIPTOMIC,cDNA
5,SRX18043706,SRP404822,MGISEQ-2000RS,SRS15550300,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,4,perfect match,not documented,missing child term,M,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476498,4,,,,,,,,,,08/04/2026,health,,4,,,,,,TRANSCRIPTOMIC,cDNA
6,SRX18043729,SRP404822,MGISEQ-2000RS,SRS15550323,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,5,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476503,5,,,,,,,,,,08/04/2026,health,,9,,,,,,TRANSCRIPTOMIC,cDNA
7,SRX18043728,SRP404822,MGISEQ-2000RS,SRS15550322,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,5,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476502,5,,,,,,,,,,08/04/2026,health,,8,,,,,,TRANSCRIPTOMIC,cDNA
8,SRX18043727,SRP404822,MGISEQ-2000RS,SRS15550321,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,5,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476501,5,,,,,,,,,,08/04/2026,health,,7,,,,,,TRANSCRIPTOMIC,cDNA
9,SRX18043730,SRP404822,MGISEQ-2000RS,SRS15550324,UBERON:0000178,blood,UBERON:0018241,prime adult stage,,Blood,6,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476504,6,,,,,,,,,,08/04/2026,health,,10,,,,,,TRANSCRIPTOMIC,cDNA


#### globin, replicates

In [9]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [10]:
#library.loc[:,'sampleAge_value'] = ''
library.loc[:,'sampleAge_unit'] = 'year'

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX18043695,SRP404822,MGISEQ-2000RS,SRS15550289,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,3,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476497,3,year,,,,,,,,,08/04/2026,health,,3,,,,,,TRANSCRIPTOMIC,cDNA
1,SRX18043684,SRP404822,MGISEQ-2000RS,SRS15550277,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,3,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476496,3,year,,,,,,,,,08/04/2026,health,,2,,,,,,TRANSCRIPTOMIC,cDNA
2,SRX18043683,SRP404822,MGISEQ-2000RS,SRS15550278,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,3,perfect match,not documented,missing child term,M,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476495,3,year,,,,,,,,,08/04/2026,health,,1,,,,,,TRANSCRIPTOMIC,cDNA
3,SRX18043726,SRP404822,MGISEQ-2000RS,SRS15550320,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,4,perfect match,not documented,missing child term,M,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476500,4,year,,,,,,,,,08/04/2026,health,,6,,,,,,TRANSCRIPTOMIC,cDNA
4,SRX18043717,SRP404822,MGISEQ-2000RS,SRS15550311,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,4,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476499,4,year,,,,,,,,,08/04/2026,health,,5,,,,,,TRANSCRIPTOMIC,cDNA
5,SRX18043706,SRP404822,MGISEQ-2000RS,SRS15550300,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,4,perfect match,not documented,missing child term,M,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476498,4,year,,,,,,,,,08/04/2026,health,,4,,,,,,TRANSCRIPTOMIC,cDNA
6,SRX18043729,SRP404822,MGISEQ-2000RS,SRS15550323,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,5,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476503,5,year,,,,,,,,,08/04/2026,health,,9,,,,,,TRANSCRIPTOMIC,cDNA
7,SRX18043728,SRP404822,MGISEQ-2000RS,SRS15550322,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,5,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476502,5,year,,,,,,,,,08/04/2026,health,,8,,,,,,TRANSCRIPTOMIC,cDNA
8,SRX18043727,SRP404822,MGISEQ-2000RS,SRS15550321,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,5,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476501,5,year,,,,,,,,,08/04/2026,health,,7,,,,,,TRANSCRIPTOMIC,cDNA
9,SRX18043730,SRP404822,MGISEQ-2000RS,SRS15550324,UBERON:0000178,blood,UBERON:0018241,prime adult stage,,Blood,6,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476504,6,year,,,,,,,,,08/04/2026,health,,10,,,,,,TRANSCRIPTOMIC,cDNA


#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [11]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-04-08'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX18043695,SRP404822,MGISEQ-2000RS,SRS15550289,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,3,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476497,3,year,,,,,,,,SAC,2026-04-08,health,,3,,,,,,TRANSCRIPTOMIC,cDNA
1,SRX18043684,SRP404822,MGISEQ-2000RS,SRS15550277,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,3,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476496,3,year,,,,,,,,SAC,2026-04-08,health,,2,,,,,,TRANSCRIPTOMIC,cDNA
2,SRX18043683,SRP404822,MGISEQ-2000RS,SRS15550278,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,3,perfect match,not documented,missing child term,M,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476495,3,year,,,,,,,,SAC,2026-04-08,health,,1,,,,,,TRANSCRIPTOMIC,cDNA
3,SRX18043726,SRP404822,MGISEQ-2000RS,SRS15550320,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,4,perfect match,not documented,missing child term,M,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476500,4,year,,,,,,,,SAC,2026-04-08,health,,6,,,,,,TRANSCRIPTOMIC,cDNA
4,SRX18043717,SRP404822,MGISEQ-2000RS,SRS15550311,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,4,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476499,4,year,,,,,,,,SAC,2026-04-08,health,,5,,,,,,TRANSCRIPTOMIC,cDNA
5,SRX18043706,SRP404822,MGISEQ-2000RS,SRS15550300,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,4,perfect match,not documented,missing child term,M,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476498,4,year,,,,,,,,SAC,2026-04-08,health,,4,,,,,,TRANSCRIPTOMIC,cDNA
6,SRX18043729,SRP404822,MGISEQ-2000RS,SRS15550323,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,5,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476503,5,year,,,,,,,,SAC,2026-04-08,health,,9,,,,,,TRANSCRIPTOMIC,cDNA
7,SRX18043728,SRP404822,MGISEQ-2000RS,SRS15550322,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,5,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476502,5,year,,,,,,,,SAC,2026-04-08,health,,8,,,,,,TRANSCRIPTOMIC,cDNA
8,SRX18043727,SRP404822,MGISEQ-2000RS,SRS15550321,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,5,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476501,5,year,,,,,,,,SAC,2026-04-08,health,,7,,,,,,TRANSCRIPTOMIC,cDNA
9,SRX18043730,SRP404822,MGISEQ-2000RS,SRS15550324,UBERON:0000178,blood,UBERON:0018241,prime adult stage,,Blood,6,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476504,6,year,,,,,,,,SAC,2026-04-08,health,,10,,,,,,TRANSCRIPTOMIC,cDNA


#### comments

In [12]:
library.loc[:,'comment'] = 'PMID: 36548828'

#### save complete file with correct columns

In [13]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX18043695,SRP404822,MGISEQ-2000RS,SRS15550289,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,3,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476497,3,year,,,,,PMID: 36548828,,,SAC,2026-04-08
1,SRX18043684,SRP404822,MGISEQ-2000RS,SRS15550277,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,3,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476496,3,year,,,,,PMID: 36548828,,,SAC,2026-04-08
2,SRX18043683,SRP404822,MGISEQ-2000RS,SRS15550278,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,3,perfect match,not documented,missing child term,M,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476495,3,year,,,,,PMID: 36548828,,,SAC,2026-04-08
3,SRX18043726,SRP404822,MGISEQ-2000RS,SRS15550320,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,4,perfect match,not documented,missing child term,M,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476500,4,year,,,,,PMID: 36548828,,,SAC,2026-04-08
4,SRX18043717,SRP404822,MGISEQ-2000RS,SRS15550311,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,4,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476499,4,year,,,,,PMID: 36548828,,,SAC,2026-04-08
5,SRX18043706,SRP404822,MGISEQ-2000RS,SRS15550300,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,4,perfect match,not documented,missing child term,M,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476498,4,year,,,,,PMID: 36548828,,,SAC,2026-04-08
6,SRX18043729,SRP404822,MGISEQ-2000RS,SRS15550323,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,5,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476503,5,year,,,,,PMID: 36548828,,,SAC,2026-04-08
7,SRX18043728,SRP404822,MGISEQ-2000RS,SRS15550322,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,5,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476502,5,year,,,,,PMID: 36548828,,,SAC,2026-04-08
8,SRX18043727,SRP404822,MGISEQ-2000RS,SRS15550321,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,5,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476501,5,year,,,,,PMID: 36548828,,,SAC,2026-04-08
9,SRX18043730,SRP404822,MGISEQ-2000RS,SRS15550324,UBERON:0000178,blood,UBERON:0018241,prime adult stage,,Blood,6,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropoda melanoleuca,SAMN31476504,6,year,,,,,PMID: 36548828,,,SAC,2026-04-08


### experiment annotations

In [14]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP404822,Ailuropoda melanoleuca Transcriptome or Gene expression,Analysis and comparison of transcriptome information of giant pandas at different ages,SRA,,,,,,,PRJNA893394,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [15]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

48

In [16]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
#experiment.loc[:,'protocol'] = my_protocol
#experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP404822,Ailuropoda melanoleuca Transcriptome or Gene expression,Analysis and comparison of transcriptome information of giant pandas at different ages,SRA,total,Bgee 1K,48,,,,PRJNA893394,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [17]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '36548828'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC9784451/'
experiment.loc[:,'DOI'] = '10.3390/vetsci9120667'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP404822,Ailuropoda melanoleuca Transcriptome or Gene expression,Analysis and comparison of transcriptome information of giant pandas at different ages,SRA,total,Bgee 1K,48,,,,PRJNA893394,36548828,https://pmc.ncbi.nlm.nih.gov/articles/PMC9784451/,10.3390/vetsci9120667,,


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [18]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [19]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [20]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [23]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [24]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
58982,SRX17520725,SRP396531,Illumina NovaSeq 6000,SRS15071290,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,blood,1.5,perfect match,not documented,missing child term,F,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,YX1,SAMN30788816,1.5,year,,,,,PMID: 36303550,Wildness training,,SAC,2026-04-08
58983,SRX17520724,SRP396531,Illumina NovaSeq 6000,SRS15071289,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,blood,1.5,perfect match,not documented,missing child term,M,,,9646,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,BX1,SAMN30788815,1.5,year,,,,,PMID: 36303550,Wildness training,,SAC,2026-04-08
58984,SRX18043695,SRP404822,MGISEQ-2000RS,SRS15550289,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,3,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropod...,SAMN31476497,3,year,,,,,PMID: 36548828,,,SAC,2026-04-08
58985,SRX18043684,SRP404822,MGISEQ-2000RS,SRS15550277,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,3,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropod...,SAMN31476496,3,year,,,,,PMID: 36548828,,,SAC,2026-04-08
58986,SRX18043683,SRP404822,MGISEQ-2000RS,SRS15550278,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,3,perfect match,not documented,missing child term,M,,,9646,,,polyA,,,Model organism or animal sample from Ailuropod...,SAMN31476495,3,year,,,,,PMID: 36548828,,,SAC,2026-04-08
58987,SRX18043726,SRP404822,MGISEQ-2000RS,SRS15550320,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,4,perfect match,not documented,missing child term,M,,,9646,,,polyA,,,Model organism or animal sample from Ailuropod...,SAMN31476500,4,year,,,,,PMID: 36548828,,,SAC,2026-04-08
58988,SRX18043717,SRP404822,MGISEQ-2000RS,SRS15550311,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Blood,4,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropod...,SAMN31476499,4,year,,,,,PMID: 36548828,,,SAC,2026-04-08


In [25]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1154,SRP387848,Comparative transcriptomics and methylomics re...,Both the giant panda (Ailuropoda melanoleuca) ...,SRA,partial,Bgee 1K,18,NEBNext Ultra RNA Library Prep Kit,full_length,,PRJNA861855,36011357,https://pmc.ncbi.nlm.nih.gov/articles/PMC9407821/,10.3390/genes13081446,,removed bisulfite seq libraries
1155,SRP396531,Gene expression profiles of wild-trained and w...,Gene expression profiles of wild-trained and w...,SRA,total,Bgee 1K,12,NEBNext Ultra RNA Library Prep Kit,full_length,,PRJNA878951,36303550,https://pmc.ncbi.nlm.nih.gov/articles/PMC9592921/,10.3389/fgene.2022.995700,34985592[PMID],
1156,SRP404822,Ailuropoda melanoleuca Transcriptome or Gene e...,Analysis and comparison of transcriptome infor...,SRA,total,Bgee 1K,48,,,,PRJNA893394,36548828,https://pmc.ncbi.nlm.nih.gov/articles/PMC9784451/,10.3390/vetsci9120667,,


### add annotations to git

In [26]:
! git pull

Already up to date.


In [27]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [ ]:
! git status

In [28]:
! git add $git_experiment_path $git_library_path

In [29]:
! git commit -m $commit_message_exp

[develop ed53629] adding annotated bulk experiment SRP404822
 2 files changed, 49 insertions(+)


In [30]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.97 KiB | 2.97 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   24e3081..ed53629  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push